# Experiment: Random Forests — OOB Error, Bagging, and Feature Subsampling

**Dataset:** `placement_predict_50k Dataset.csv`  
**Target:** `PlacementStatus`

This notebook compares Decision Tree, Bagging, and Random Forest, calculates OOB Error, and studies feature subsampling and number of trees.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


In [ ]:
uploaded = files.upload()


In [ ]:
# Automatically identify the uploaded CSV file
csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]

if not csv_files:
    raise FileNotFoundError("Please upload a CSV dataset.")

file_name = csv_files[0]
df = pd.read_csv(file_name)

print("Using file:", file_name)
print("Dataset Shape:", df.shape)
df.head()


In [ ]:
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)


In [ ]:
print("Missing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())


In [ ]:
print(df["PlacementStatus"].value_counts())

plt.figure(figsize=(6, 4))
df["PlacementStatus"].value_counts().plot(kind="bar")
plt.title("Placement Status Distribution")
plt.xlabel("Placement Status")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.show()


## Prepare Features and Target

`PlacementStatus` is the target.

The following columns are excluded:
- `StudentID` — identifier only
- `Salary Package` — can cause target leakage because it is an outcome-related field
- `IsAnomaly` — excluded from the prediction experiment


In [ ]:
X = df.drop(
    columns=["PlacementStatus", "StudentID", "Salary Package", "IsAnomaly"],
    errors="ignore"
)

y = df["PlacementStatus"]

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)


In [ ]:
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)


In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## 1. Decision Tree Baseline

In [ ]:
decision_tree = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]
)

decision_tree.fit(X_train, y_train)
dt_predictions = decision_tree.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_predictions)

print("Decision Tree Accuracy:", dt_accuracy)


In [ ]:
print("Decision Tree Classification Report:")
print(classification_report(y_test, dt_predictions))


## 2. Bagging

In [ ]:
bagging_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", BaggingClassifier(
            estimator=DecisionTreeClassifier(random_state=42),
            n_estimators=100,
            max_samples=1.0,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

bagging_model.fit(X_train, y_train)
bagging_predictions = bagging_model.predict(X_test)

bagging_accuracy = accuracy_score(y_test, bagging_predictions)

print("Bagging Accuracy:", bagging_accuracy)


In [ ]:
print("Bagging Classification Report:")
print(classification_report(y_test, bagging_predictions))


## 3. Random Forest with OOB Error

In [ ]:
random_forest = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_features="sqrt",
            bootstrap=True,
            oob_score=True,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

random_forest.fit(X_train, y_train)
rf_predictions = random_forest.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_predictions)

print("Random Forest Accuracy:", rf_accuracy)


In [ ]:
rf_classifier = random_forest.named_steps["classifier"]

rf_oob_score = rf_classifier.oob_score_
rf_oob_error = 1 - rf_oob_score

print("OOB Score:", rf_oob_score)
print("OOB Error:", rf_oob_error)


In [ ]:
print("Random Forest Classification Report:")
print(classification_report(y_test, rf_predictions))


## 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, rf_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 5. Feature Subsampling

Compare different `max_features` settings.


In [ ]:
feature_settings = ["sqrt", "log2", 0.5, 0.7, 1.0]
feature_results = []

for setting in feature_settings:
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(
                n_estimators=100,
                max_features=setting,
                bootstrap=True,
                oob_score=True,
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    oob_score = model.named_steps["classifier"].oob_score_
    oob_error = 1 - oob_score

    feature_results.append({
        "max_features": str(setting),
        "Accuracy": accuracy,
        "OOB Score": oob_score,
        "OOB Error": oob_error
    })

feature_results_df = pd.DataFrame(feature_results)
feature_results_df


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    feature_results_df["max_features"],
    feature_results_df["Accuracy"],
    marker="o",
    label="Accuracy"
)

plt.plot(
    feature_results_df["max_features"],
    feature_results_df["OOB Score"],
    marker="s",
    label="OOB Score"
)

plt.xlabel("max_features")
plt.ylabel("Score")
plt.title("Effect of Feature Subsampling")
plt.legend()
plt.grid()
plt.show()


## 6. Effect of Number of Trees

In [ ]:
tree_numbers = [10, 25, 50, 100, 150, 200]
tree_results = []

for n in tree_numbers:
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", RandomForestClassifier(
                n_estimators=n,
                max_features="sqrt",
                bootstrap=True,
                oob_score=True,
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    oob_score = model.named_steps["classifier"].oob_score_
    oob_error = 1 - oob_score

    tree_results.append({
        "Number of Trees": n,
        "Accuracy": accuracy,
        "OOB Score": oob_score,
        "OOB Error": oob_error
    })

tree_results_df = pd.DataFrame(tree_results)
tree_results_df


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    tree_results_df["Number of Trees"],
    tree_results_df["OOB Error"],
    marker="o"
)

plt.xlabel("Number of Trees")
plt.ylabel("OOB Error")
plt.title("OOB Error vs Number of Trees")
plt.grid()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    tree_results_df["Number of Trees"],
    tree_results_df["Accuracy"],
    marker="o"
)

plt.xlabel("Number of Trees")
plt.ylabel("Accuracy")
plt.title("Random Forest Accuracy vs Number of Trees")
plt.grid()
plt.show()


## 7. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Bagging", "Random Forest"],
    "Accuracy": [dt_accuracy, bagging_accuracy, rf_accuracy]
})

comparison


In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(comparison["Model"], comparison["Accuracy"])
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
plt.ylim(0, 1)
plt.show()


## 8. Best Feature Subsampling Setting

In [ ]:
best_row = feature_results_df.loc[
    feature_results_df["Accuracy"].idxmax()
]

print("Best Feature Subsampling Setting:")
print(best_row)


## 9. Final Results

In [ ]:
print("========== FINAL RESULTS ==========")
print(f"Decision Tree Accuracy: {dt_accuracy:.4f}")
print(f"Bagging Accuracy:       {bagging_accuracy:.4f}")
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest OOB Score: {rf_oob_score:.4f}")
print(f"Random Forest OOB Error: {rf_oob_error:.4f}")
print("===================================")


## Conclusion

Random Forest combines **bagging** with **random feature subsampling**. Bagging reduces variance by training multiple decision trees on bootstrap samples. Random Forest additionally considers a random subset of features at each split, increasing diversity among trees.

The OOB score provides an internal performance estimate using samples not selected for a particular bootstrap sample, while OOB Error is calculated as:

**OOB Error = 1 − OOB Score**

The experiment also studies how the number of trees and feature subsampling affect Random Forest performance.
